# Components lab 01: ports and component basics

A component is an event target. A port is a named boundary on that component. A connection turns `source.out.transmit(payload)` into a scheduled event for the target component.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

from simyuj.components import Port, PortDelivery, PortDirection, PortKind, connect_ports
from simyuj.engine import Component, Event, Timeline

## 1. Make two tiny components

The sender owns an egress port. The receiver owns an ingress port and records what arrives.

In [ ]:
@dataclass(slots=True)
class LabSender(Component):
    component_id: str
    output_port: Port = field(init=False)
    sent: list[tuple[int, object]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.output_port = Port(
            name='out',
            owner=self,
            owner_id=self.component_id,
            port_kind=PortKind.CLASSICAL,
            direction=PortDirection.EGRESS,
        )

    def handle_event(self, event, timeline) -> None:
        if event.action == 'send_now':
            payload = event.payload_ref
            self.output_port.connection.transmit(payload, timeline, source=self)
            self.sent.append((timeline.current_time, payload))
            return
        raise ValueError(event.action)


In [ ]:
@dataclass(slots=True)
class LabReceiver(Component):
    component_id: str
    input_port: Port = field(init=False)
    inbox: list[tuple[int, str, object, str]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port(
            name='in',
            owner=self,
            owner_id=self.component_id,
            port_kind=PortKind.CLASSICAL,
            direction=PortDirection.INGRESS,
        )

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.inbox.append(
            (
                timeline.current_time,
                event.action,
                delivery.payload,
                delivery.connection_id,
            )
        )


In [ ]:
sender = LabSender('alice.ctrl')
receiver = LabReceiver('bob.ctrl')

print('sender port:', sender.output_port.owner_id, sender.output_port.name, sender.output_port.direction.value)
print('receiver port:', receiver.input_port.owner_id, receiver.input_port.name, receiver.input_port.direction.value)
print('connected before wiring:', sender.output_port.is_connected, receiver.input_port.is_connected)

## 2. Wire the boundary

A connection installs itself on both ports and remembers the target action for delivery events.

In [ ]:
connection = connect_ports(
    sender.output_port,
    receiver.input_port,
    target_action='receive_packet',
)

print('connection id:', connection.connection_id)
print('connected after wiring:', sender.output_port.is_connected, receiver.input_port.is_connected)
print('target action:', connection.target_action)

In [ ]:
timeline = Timeline(master_seed=10)
event = connection.transmit({'basis': 'Z', 'round': 1}, timeline, time=7)

print('scheduled event id:', event.event_id)
print('scheduled target is receiver:', event.target_ref is receiver)
print('event action:', event.action)
print('event time:', event.time)
print('event meta:', event.meta)
print('receiver before running:', receiver.inbox)

In [ ]:
timeline.run_until(7)

print('receiver after running:')
for row in receiver.inbox:
    print(row)

## 3. Let a component transmit from inside `handle_event`

Now the timeline wakes the sender. The sender uses its connected output port; it does not call the receiver directly.

In [ ]:
sender = LabSender('alice.ctrl')
receiver = LabReceiver('bob.ctrl')
connect_ports(sender.output_port, receiver.input_port, target_action='receive_packet')
timeline = Timeline(master_seed=10)

timeline.schedule(
    Event(
        time=3,
        target_ref=sender,
        action='send_now',
        payload_ref={'basis': 'X', 'round': 2},
    )
)

first = timeline.run_one_step()
print('first batch:', first)
print('sender log:', sender.sent)
print('receiver after sender batch:', receiver.inbox)

second = timeline.run_one_step()
print('second batch:', second)
print('receiver after delivery batch:', receiver.inbox)

## 4. Delivery time is absolute

`connection.transmit(..., time=...)` takes an absolute simulation tick. It is not a delay.

In [ ]:
sender = LabSender('timed.sender')
receiver = LabReceiver('timed.receiver')
connection = connect_ports(sender.output_port, receiver.input_port, target_action='receive_packet')
timeline = Timeline()

connection.transmit('arrives at tick 5', timeline, time=5)
connection.transmit('arrives at tick 2', timeline, time=2)

summaries = timeline.run_until_empty()
print('batch summaries:', summaries)
print('arrival order:', receiver.inbox)

## 5. Port kind is part of the boundary

Classical and quantum ports do not connect to each other. That error is useful: it catches a wrong-plane wiring mistake before simulation starts.

In [ ]:
@dataclass(slots=True)
class QuantumReceiver(Component):
    component_id: str
    input_port: Port = field(init=False)

    def __post_init__(self) -> None:
        self.input_port = Port(
            name='qin',
            owner=self,
            owner_id=self.component_id,
            port_kind=PortKind.QUANTUM,
            direction=PortDirection.INGRESS,
        )

    def handle_event(self, event, timeline) -> None:
        pass

sender = LabSender('classical.sender')
quantum_receiver = QuantumReceiver('quantum.receiver')

try:
    connect_ports(sender.output_port, quantum_receiver.input_port, target_action='receive_signal')
except ValueError as exc:
    print('kind mismatch:', exc)

## Keep this model in your head

Components handle events. Ports describe boundaries. Connections schedule delivery events. `PortDelivery` is the thing the receiving component reads to know which payload crossed which boundary.